In [17]:
import pandas as pd

# Load cleaned dataset
df = pd.read_csv("cleaned_data.csv")

# Display first 5 rows
df.head()

# Regression target
y_reg = df["charges"]

# Classification target
y_clf = (df["charges"] > df["charges"].median()).astype(int)

# Feature matrix
X = df.drop(columns=["charges"])

X = pd.get_dummies(
    X,
    drop_first=True
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Regression split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X,
    y_reg,
    test_size=0.2,
    random_state=42
)
# Classification split
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X,
    y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)
scaler = StandardScaler()
X_train_clf_scaled = scaler.fit_transform(X_train_clf)
X_test_clf_scaled = scaler.transform(X_test_clf)

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Baseline Decision Tree
dt_baseline = DecisionTreeClassifier(random_state=42)

dt_baseline.fit(X_train_clf_scaled, y_train_clf)

# Predictions
y_train_pred = dt_baseline.predict(X_train_clf_scaled)
y_test_pred = dt_baseline.predict(X_test_clf_scaled)

# Accuracy
train_accuracy = accuracy_score(y_train_clf, y_train_pred)
test_accuracy = accuracy_score(y_test_clf, y_test_pred)
accuracy_gap = train_accuracy - test_accuracy

print(f"Training Accuracy : {train_accuracy:.4f}")
print(f"Test Accuracy     : {test_accuracy:.4f}")
print(f"Accuracy Gap      : {accuracy_gap:.4f}")

#Task 2: Controlled decision tree
dt_controlled = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

# Train the model
dt_controlled.fit(
    X_train_clf_scaled,
    y_train_clf
)

# Predictions on training data
y_train_pred_controlled = dt_controlled.predict(
    X_train_clf_scaled
)
# Predictions on test data
y_test_pred_controlled = dt_controlled.predict(
    X_test_clf_scaled
)

# Training Accuracy
train_accuracy_controlled = accuracy_score(
    y_train_clf,
    y_train_pred_controlled
)
# Test Accuracy
test_accuracy_controlled = accuracy_score(
    y_test_clf,
    y_test_pred_controlled
)
# Accuracy Gap
accuracy_gap_controlled = (
    train_accuracy_controlled -
    test_accuracy_controlled
)

comparison = pd.DataFrame({
    "Model": [
        "Baseline Decision Tree",
        "Controlled Decision Tree"
    ],
    "Training Accuracy": [
        train_accuracy,
        train_accuracy_controlled
    ],
    "Test Accuracy": [
        test_accuracy,
        test_accuracy_controlled
    ],
    "Accuracy Gap": [
        accuracy_gap,
        accuracy_gap_controlled
    ]
})

print(comparison.round(4))

print("=" * 60)
print("Controlled Decision Tree Results")
print("=" * 60)

print(f"Training Accuracy : {train_accuracy_controlled:.4f}")
print(f"Test Accuracy     : {test_accuracy_controlled:.4f}")
print(f"Accuracy Gap      : {accuracy_gap_controlled:.4f}")

#Decision Tree using Gini Impurity
dt_gini = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

dt_gini.fit(
    X_train_clf_scaled,
    y_train_clf
)

#Decision Tree using Entropy
dt_entropy = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)

dt_entropy.fit(
    X_train_clf_scaled,
    y_train_clf
)

# Predictions
y_pred_gini = dt_gini.predict(X_test_clf_scaled)

y_pred_entropy = dt_entropy.predict(X_test_clf_scaled)

gini_accuracy = accuracy_score(
    y_test_clf,
    y_pred_gini
)

entropy_accuracy = accuracy_score(
    y_test_clf,
    y_pred_entropy
)

comparison = pd.DataFrame({
    "Criterion": [
        "Gini",
        "Entropy"
    ],
    "Test Accuracy": [
        gini_accuracy,
        entropy_accuracy
    ]
})

print(comparison.round(4))

print("=" * 50)
print("Gini vs Entropy Comparison")
print("=" * 50)

print(f"Gini Test Accuracy     : {gini_accuracy:.4f}")
print(f"Entropy Test Accuracy  : {entropy_accuracy:.4f}")

#Task 4: Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

# Train the model
rf_model.fit(
    X_train_clf_scaled,
    y_train_clf
)
# Class predictions
y_train_pred_rf = rf_model.predict(
    X_train_clf_scaled
)

y_test_pred_rf = rf_model.predict(
    X_test_clf_scaled
)

# Probability predictions
y_prob_rf = rf_model.predict_proba(
    X_test_clf_scaled
)[:,1]

# Training Accuracy
rf_train_accuracy = accuracy_score(
    y_train_clf,
    y_train_pred_rf
)

# Test Accuracy
rf_test_accuracy = accuracy_score(
    y_test_clf,
    y_test_pred_rf
)

# ROC AUC
rf_auc = roc_auc_score(
    y_test_clf,
    y_prob_rf
)

print("="*55)
print("Random Forest Results")
print("="*55)

print(f"Training Accuracy : {rf_train_accuracy:.4f}")
print(f"Test Accuracy     : {rf_test_accuracy:.4f}")
print(f"ROC-AUC           : {rf_auc:.4f}")

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

top5_features = feature_importance.head(5)

print(top5_features)

#Gradient Boosting Classifier
from sklearn.ensemble import GradientBoostingClassifier
#Create Gradient Boosting model
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Train the model
gb_model.fit(
    X_train_clf_scaled,
    y_train_clf
)

# Predict class labels
y_train_pred_gb = gb_model.predict(X_train_clf_scaled)
y_test_pred_gb = gb_model.predict(X_test_clf_scaled)

# Predict probabilities
y_prob_gb = gb_model.predict_proba(X_test_clf_scaled)[:, 1]

# Training Accuracy
gb_train_accuracy = accuracy_score(
    y_train_clf,
    y_train_pred_gb
)

# Test Accuracy
gb_test_accuracy = accuracy_score(
    y_test_clf,
    y_test_pred_gb
)

# ROC-AUC
gb_auc = roc_auc_score(
    y_test_clf,
    y_prob_gb
)

print("=" * 55)
print("Gradient Boosting Results")
print("=" * 55)

print(f"Training Accuracy : {gb_train_accuracy:.4f}")
print(f"Test Accuracy     : {gb_test_accuracy:.4f}")
print(f"ROC-AUC           : {gb_auc:.4f}")

#Feature ablation study
# Bottom 5 features
bottom5_features = feature_importance.tail(5)
print(bottom5_features)
# Features to remove
remove_features = bottom5_features["Feature"].tolist()

print("Removed Features:")
print(remove_features)

# Keep only remaining features
X_train_reduced = X_train_clf.drop(columns=remove_features)
X_test_reduced = X_test_clf.drop(columns=remove_features)

scaler_reduced = StandardScaler()

X_train_reduced_scaled = scaler_reduced.fit_transform(X_train_reduced)
X_test_reduced_scaled = scaler_reduced.transform(X_test_reduced)
#Train another random forest
rf_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_reduced.fit(X_train_reduced_scaled,y_train_clf)
#Predict probabilities
y_prob_reduced = rf_reduced.predict_proba(X_test_reduced_scaled)[:,1]

#Compute ROC-AUC for reduced model
reduced_auc = roc_auc_score(y_test_clf,y_prob_reduced)
#Compare Models
comparison = pd.DataFrame({
    "Model": ["Full Random Forest", "Reduced Random Forest"],
    "ROC-AUC": [rf_auc,reduced_auc]
})

print(comparison)
#Difference
difference = rf_auc - reduced_auc
print(f"AUC Difference : {difference:.4f}")

#Task 5: Cross-validation
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Logistic Regression (same as Part 2)
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Controlled Decision Tree (Task 2)
dt_controlled = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

# Random Forest (Task 4)
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

# Gradient Boosting (Task 4a)
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

#Create cross-vaidation object
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

logistic_auc = cross_val_score(
    logistic_model,
    X_train_clf_scaled,
    y_train_clf,
    cv=cv,
    scoring="roc_auc"
)
dt_auc = cross_val_score(
    dt_controlled,
    X_train_clf_scaled,
    y_train_clf,
    cv=cv,
    scoring="roc_auc"
)

rf_auc = cross_val_score(
    rf_model,
    X_train_clf_scaled,
    y_train_clf,
    cv=cv,
    scoring="roc_auc"
)

gb_auc = cross_val_score(
    gb_model,
    X_train_clf_scaled,
    y_train_clf,
    cv=cv,
    scoring="roc_auc"
)

#Create comparison table
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Controlled Decision Tree",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Mean ROC-AUC": [
        logistic_auc.mean(),
        dt_auc.mean(),
        rf_auc.mean(),
        gb_auc.mean()
    ],
    "Std ROC-AUC": [
        logistic_auc.std(),
        dt_auc.std(),
        rf_auc.std(),
        gb_auc.std()
    ]
})

comparison = comparison.sort_values(
    by="Mean ROC-AUC",
    ascending=False
)

print(comparison.round(4))

# Task 6: Hyperparameter tuning for Random Forest
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

param_grid = {
    "randomforestclassifier__n_estimators": [50, 100, 200],
    "randomforestclassifier__max_depth": [5, 10, None],
    "randomforestclassifier__min_samples_leaf": [1, 5]
}
#Cross Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

#Create Grid searchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

#Use the unscaled training data for grid search
grid_search.fit(X_train_clf, y_train_clf)
#Best Parameters
print("=" * 60)
print("Best Parameters")
print("=" * 60)

print(grid_search.best_params_)
# Best ROC-AUC
print()

print("=" * 60)
print("Best Cross Validation ROC-AUC")
print("=" * 60)

print(grid_search.best_score_)
# Best Pipeline
best_pipeline = grid_search.best_estimator_
# Model Configuration
total_models = (
    len(param_grid["randomforestclassifier__n_estimators"])
    * len(param_grid["randomforestclassifier__max_depth"])
    * len(param_grid["randomforestclassifier__min_samples_leaf"])
)

total_evaluations = total_models * 5

print(f"Total Model Configurations : {total_models}")
print(f"Total CV Evaluations       : {total_evaluations}")

# Task 7: Manual Learning Curve
import numpy as np
from sklearn.metrics import roc_auc_score

# Define training set sizes
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
results = []

for f in fractions:

    # Number of rows to use
    n_rows = int(f * len(X_train_clf))

    # Training subset
    X_subset = X_train_clf.iloc[:n_rows]
    y_subset = y_train_clf.iloc[:n_rows]

    # Train the best pipeline
    best_pipeline.fit(X_subset, y_subset)

    # Training probabilities
    train_prob = best_pipeline.predict_proba(X_subset)[:, 1]

    # Test probabilities
    test_prob = best_pipeline.predict_proba(X_test_clf)[:, 1]

    # Compute ROC-AUC
    train_auc = roc_auc_score(y_subset, train_prob)
    test_auc = roc_auc_score(y_test_clf, test_prob)

    results.append([f, train_auc, test_auc])

#Display results
learning_curve = pd.DataFrame(
    results,
    columns=[
        "Training Fraction",
        "Training AUC",
        "Test AUC"
    ]
)

print("\nManual Learning Curve")
print("=" * 60)

print(learning_curve.round(4))

# Task 8: Serialize the best model
import joblib
# Save the trained pipeline
joblib.dump(best_pipeline, "best_model.pkl")
print("Model saved successfully as best_model.pkl")
# Load the saved model
loaded_model = joblib.load("best_model.pkl")
print("Model loaded successfully!")

new_data = pd.DataFrame({
    "age": [30, 55],
    "bmi": [24.5, 36.8],
    "children": [1, 3],
    "sex_male": [1, 0],
    "smoker_yes": [0, 1],
    "region_northwest": [1, 0],
    "region_southeast": [0, 1],
    "region_southwest": [0, 0]
})

# Make predictions
predictions = loaded_model.predict(new_data)
print("Predictions:")
print(predictions)


Training Accuracy : 0.9921
Test Accuracy     : 0.8838
Accuracy Gap      : 0.1083
                      Model  Training Accuracy  Test Accuracy  Accuracy Gap
0    Baseline Decision Tree             0.9921         0.8838        0.1083
1  Controlled Decision Tree             0.9305         0.9120        0.0185
Controlled Decision Tree Results
Training Accuracy : 0.9305
Test Accuracy     : 0.9120
Accuracy Gap      : 0.0185
  Criterion  Test Accuracy
0      Gini         0.9085
1   Entropy         0.9085
Gini vs Entropy Comparison
Gini Test Accuracy     : 0.9085
Entropy Test Accuracy  : 0.9085
Random Forest Results
Training Accuracy : 0.9595
Test Accuracy     : 0.9120
ROC-AUC           : 0.9312
      Feature  Importance
0         age    0.504755
4  smoker_yes    0.295935
1         bmi    0.118124
2    children    0.043603
3    sex_male    0.014345
Gradient Boosting Results
Training Accuracy : 0.9481
Test Accuracy     : 0.9190
ROC-AUC           : 0.9422
            Feature  Importance
2      